# LR3 — Neural Network Classification (TensorFlow/Keras)

Dataset: adult.csv (Census Income)

Target: income (<=50K / >50K) — binary classification

1. Import
2. Load data
3. EDA
4. Preprocessing
5. Build, compile & train model
6. Visualize training
7. Evaluate model

In [ ]:
# 1 - Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# 2 - Load data
df = pd.read_csv("adult.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# 3 - EDA
print("Missing values:")
print(df.isnull().sum())
print(f"\nTarget distribution:")
print(df['income'].value_counts())
print(f"\nNumeric stats:")
df.describe()

In [ ]:
# 3 (cont) - Visualizations
num_cols = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, edgecolor='black')
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='income', y=col, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
income_counts = df['income'].value_counts()
axes[0].bar(income_counts.index, income_counts.values, color=['#2196F3', '#FF5722'])
axes[0].set_title('Income distribution')
axes[1].pie(income_counts.values, labels=income_counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Income (%)')
plt.tight_layout()
plt.show()

In [ ]:
# 4 - Preprocessing

df = df.drop(columns=['fnlwgt'])

# Encode target
df['income'] = (df['income'] == '>50K').astype(int)

# Encode categorical features
cat_features = df.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Feature matrix
feature_cols = [c for c in df.columns if c != 'income']
X = df[feature_cols].values
y = df['income'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target: 0={sum(y==0)}, 1={sum(y==1)}")

In [ ]:
# 5 - Build, compile & train model

input_shape = X_train.shape[1:]

model = keras.models.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=input_shape),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 5 (cont) - Train

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 6 - Visualize training

history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['loss'], label='Train loss')
axes[0].plot(history_df['val_loss'], label='Val loss')
axes[0].set_title('Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['accuracy'], label='Train accuracy')
axes[1].plot(history_df['val_accuracy'], label='Val accuracy')
axes[1].set_title('Accuracy over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7 - Evaluate model

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['<=50K', '>50K'],
            yticklabels=['<=50K', '>50K'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 7 (cont) - Improved model with more layers + dropout

model_2 = keras.models.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=input_shape),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model_2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop_2 = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history_2 = model_2.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop_2],
    verbose=1
)

test_loss_2, test_acc_2 = model_2.evaluate(X_test, y_test)
print(f"\nImproved model — Test loss: {test_loss_2:.4f}, Test accuracy: {test_acc_2:.4f}")
print(f"Baseline model — Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")

In [ ]:
# 7 (cont) - Compare models

history_df_2 = pd.DataFrame(history_2.history)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss comparison
axes[0].plot(history_df['val_loss'], label='Baseline val_loss')
axes[0].plot(history_df_2['val_loss'], label='Improved val_loss')
axes[0].set_title('Validation Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy comparison
axes[1].plot(history_df['val_accuracy'], label='Baseline val_acc')
axes[1].plot(history_df_2['val_accuracy'], label='Improved val_acc')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Bar chart
models = ['Baseline', 'Improved']
accs = [test_acc, test_acc_2]
bars = axes[2].bar(models, accs, color=['steelblue', 'forestgreen'])
axes[2].set_ylim(0.7, 0.95)
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Test Accuracy Comparison')
for bar, acc in zip(bars, accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{acc:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Improved model confusion matrix
y_pred_2 = (model_2.predict(X_test) > 0.5).astype(int).flatten()
print("\nImproved Model Classification Report:")
print(classification_report(y_test, y_pred_2, target_names=['<=50K', '>50K']))